# V1: Without education-job features

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import random

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
companies = pd.read_csv("companies.csv")
portfolio_edges = pd.read_csv("portfolio_edges.csv")
feature_matrix = pd.read_csv("feature_matrix.csv")

print("companies shape:", companies.shape)
print("portfolio_edges shape:", portfolio_edges.shape)
print("feature_matrix shape:", feature_matrix.shape)

companies shape: (6704, 22)
portfolio_edges shape: (461543, 6)
feature_matrix shape: (6704, 37)


In [3]:
companies = companies[companies["is_success"].notna()].copy()
companies["is_success"] = companies["is_success"].astype(int)

feature_matrix = feature_matrix[feature_matrix["is_success"].notna()].copy()
feature_matrix["is_success"] = feature_matrix["is_success"].astype(int)

print("labeled companies:", companies.shape)
print("labeled feature_matrix:", feature_matrix.shape)

labeled companies: (6593, 22)
labeled feature_matrix: (6593, 37)


In [5]:
G = nx.Graph()

labeled_company_ids = set(companies["company_uuid"].astype(str))

for _, row in portfolio_edges.iterrows():
    investor_id = "inv_" + str(row["vc_uuid"])
    company_id = str(row["portfolio_company_uuid"])
    G.add_edge(investor_id, company_id)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

Graph nodes: 160243
Graph edges: 353034


In [6]:
def random_walk(graph, start_node, walk_length=10):
    walk = [start_node]
    
    for _ in range(walk_length - 1):
        current_node = walk[-1]
        neighbors = list(graph.neighbors(current_node))
        
        if len(neighbors) == 0:
            break
            
        next_node = random.choice(neighbors)
        walk.append(next_node)
    
    return [str(node) for node in walk]

In [18]:
random.seed(42)
np.random.seed(42)

walks = []
all_nodes = list(G.nodes())

num_walks = 8
walk_length = 20

for _ in range(num_walks):
    random.shuffle(all_nodes)
    for node in all_nodes:
        walks.append(random_walk(G, node, walk_length=walk_length))

print("Number of walks:", len(walks))
print("Example walk:", walks[0][:10])

Number of walks: 1281944
Example walk: ['0c6c101e-53f6-49c9-a57c-7f98a08301e4', 'inv_796e4250-e66a-1c7b-6ccf-5a4fbd3b1936', 'f08a3bfa-32a0-0015-645c-d893f2a2c2ae', 'inv_9183d3fb-c801-bc11-1594-04850e47cf60', 'ca9a4738-54ec-435a-95c9-1faabdf24b7d', 'inv_73633ee4-ea65-2967-6c5d-9b5fec7d2d5e', '9dc9cd36-acc4-43e0-96e9-b57eff2dbbed', 'inv_73633ee4-ea65-2967-6c5d-9b5fec7d2d5e', '73c10c11-d3e8-4c87-8378-14d959b3e7e1', 'inv_73633ee4-ea65-2967-6c5d-9b5fec7d2d5e']


In [19]:
embedding_model = Word2Vec(
    sentences=walks,
    vector_size=64,
    window=8,
    min_count=0,
    sg=1,
    workers=1,
    epochs=5,
    negative=10,
    seed=42
)

print("Embedding dimension:", embedding_model.vector_size)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Embedding dimension: 64


In [20]:
embedding_dict = {}

company_ids_for_embedding = feature_matrix["company_uuid"].astype(str).unique()

for company_id in company_ids_for_embedding:
    if company_id in embedding_model.wv:
        embedding_dict[company_id] = embedding_model.wv[company_id]
    else:
        embedding_dict[company_id] = np.zeros(embedding_model.vector_size)

emb_df = pd.DataFrame.from_dict(embedding_dict, orient="index")
emb_df.index.name = "company_uuid"
emb_df.reset_index(inplace=True)

emb_df.columns = ["company_uuid"] + [f"emb_{i}" for i in range(emb_df.shape[1] - 1)]

print("emb_df shape:", emb_df.shape)
emb_df.head()

emb_df shape: (6593, 65)


,company_uuid,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63
0,fdd4bfe1-5f7b-e403-d100-91e9ec01f903,-0.500336,-0.745521,0.620051,0.256790,0.147041,-0.270557,0.044816,-0.100876,-0.911015,...,-0.675585,-0.024210,0.662975,-0.318069,0.019751,-0.244962,-0.180128,0.310385,0.003881,0.668118
1,1ebaa59b-7035-75d2-316d-e7b35b8a82ed,-0.291896,-0.842345,1.027760,-0.278997,-0.541694,-0.612289,-0.101613,-0.168379,-0.348397,...,-0.940329,-0.612759,0.881294,0.112820,0.503059,-0.371877,0.601466,0.688944,-0.384988,0.396181
2,4361b5e3-697b-325f-63e7-1814d63b865f,-0.411641,-0.512436,-0.435950,0.302411,-0.422102,0.030906,-0.257594,-0.629021,-0.887230,...,-1.083628,-0.432514,0.260129,-0.186262,-0.276814,-0.486926,-0.098476,0.154913,-0.139873,0.902288
3,5e298e78-0999-fe40-4b90-c21ed1c90786,0.171323,-0.520012,0.062282,0.346763,0.232692,0.017975,-0.315731,0.316005,-0.807183,...,-1.056073,-0.358054,0.625099,0.595592,-0.002246,-0.123182,-0.601823,-0.412057,1.092736,-0.363344
4,16c38a41-e42c-e440-f372-d5f382be7662,0.651558,-1.276370,0.076650,0.210075,-0.484757,-0.454397,0.219984,0.198947,-0.497910,...,-0.502830,0.333481,0.624781,-0.208279,1.016847,-0.102323,-0.375099,0.201036,-0.098954,0.595733


In [21]:
leakage_cols = [
    "funding_total_usd",
    "log_funding",
    "num_funding_rounds",
    "company_age_months"
]

drop_cols = ["is_success"] + [col for col in leakage_cols if col in feature_matrix.columns]

feature_only_df = feature_matrix.drop(columns=drop_cols).copy()

for col in feature_only_df.columns:
    if feature_only_df[col].dtype == bool:
        feature_only_df[col] = feature_only_df[col].astype(int)

print("feature_only_df shape:", feature_only_df.shape)
print(feature_only_df.dtypes.head())

feature_only_df shape: (6593, 32)
company_uuid          object
employees_ordinal    float64
num_founders           int64
founder_has_phd        int64
founder_has_mba        int64
dtype: object


In [22]:
model6_df = feature_matrix[["company_uuid", "is_success"]].copy()

model6_df = model6_df.merge(feature_only_df, on="company_uuid", how="left")
model6_df = model6_df.merge(emb_df, on="company_uuid", how="left")

model6_df.fillna(0, inplace=True)

print("model6_df shape:", model6_df.shape)
model6_df.head()

model6_df shape: (6593, 97)


,company_uuid,is_success,employees_ordinal,num_founders,founder_has_phd,founder_has_mba,max_founder_prior_jobs,avg_founder_prior_jobs,team_size,c_suite_count,...,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63
0,fdd4bfe1-5f7b-e403-d100-91e9ec01f903,0,2.0,4,0,0,6,3.750000,0,0,...,-0.675585,-0.024210,0.662975,-0.318069,0.019751,-0.244962,-0.180128,0.310385,0.003881,0.668118
1,1ebaa59b-7035-75d2-316d-e7b35b8a82ed,1,7.0,3,1,0,11,6.000000,21,6,...,-0.940329,-0.612759,0.881294,0.112820,0.503059,-0.371877,0.601466,0.688944,-0.384988,0.396181
2,4361b5e3-697b-325f-63e7-1814d63b865f,1,5.0,2,0,0,5,4.000000,6,3,...,-1.083628,-0.432514,0.260129,-0.186262,-0.276814,-0.486926,-0.098476,0.154913,-0.139873,0.902288
3,5e298e78-0999-fe40-4b90-c21ed1c90786,0,2.0,3,0,0,2,1.666667,3,1,...,-1.056073,-0.358054,0.625099,0.595592,-0.002246,-0.123182,-0.601823,-0.412057,1.092736,-0.363344
4,16c38a41-e42c-e440-f372-d5f382be7662,0,2.0,1,0,1,1,1.000000,0,0,...,-0.502830,0.333481,0.624781,-0.208279,1.016847,-0.102323,-0.375099,0.201036,-0.098954,0.595733


In [23]:
X_model6 = model6_df.drop(columns=["company_uuid", "is_success"]).copy()
y_model6 = model6_df["is_success"].astype(int).copy()

X_model6.columns = X_model6.columns.astype(str)

print("X_model6 shape:", X_model6.shape)
print("y_model6 shape:", y_model6.shape)

X_model6 shape: (6593, 95)
y_model6 shape: (6593,)


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model6,
    y_model6,
    test_size=0.2,
    random_state=42,
    stratify=y_model6
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5274, 95)
Test shape: (1319, 95)


In [25]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
model6_clf = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

model6_clf.fit(X_train_scaled, y_train)

y_pred_prob = model6_clf.predict_proba(X_test_scaled)[:, 1]

In [27]:
model6_roc_auc = roc_auc_score(y_test, y_pred_prob)
model6_pr_auc = average_precision_score(y_test, y_pred_prob)

print("=== Model: Node2Vec ===")
print("ROC-AUC:", model6_roc_auc)
print("PR-AUC :", model6_pr_auc)

=== Model: Node2Vec ===
ROC-AUC: 0.814881264115741
PR-AUC : 0.5499746509934309


In [28]:
print("Positive rate in full data :", y_model6.mean())
print("Positive rate in train data:", y_train.mean())
print("Positive rate in test data :", y_test.mean())

Positive rate in full data : 0.16927043834369787
Positive rate in train data: 0.16932119833143724
Positive rate in test data : 0.16906747536012132


In [29]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    classification_report
)
import pandas as pd
import numpy as np

y_pred = (y_pred_prob >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_prob)
pr_auc = average_precision_score(y_test, y_pred_prob)
accuracy = accuracy_score(y_test, y_pred)

baseline_pr = y_test.mean()

print(f"ROC-AUC: {roc_auc:.4f} (test)")
print(f"PR-AUC: {pr_auc:.4f} (test) | Baseline: {baseline_pr:.3f}")
print(f"Accuracy: {accuracy*100:.1f}%")

ROC-AUC: 0.8149 (test)
PR-AUC: 0.5500 (test) | Baseline: 0.169
Accuracy: 76.4%


In [30]:
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=["Not Success (0)", "Success (1)"],
    output_dict=True
)

report_df = pd.DataFrame(report_dict).T

report_df = report_df[["precision", "recall", "f1-score", "support"]]

report_df = report_df.loc[
    ["Not Success (0)", "Success (1)", "accuracy", "weighted avg"]
].copy()

report_df["support"] = report_df["support"].astype(int)

report_df = report_df.round({
    "precision": 2,
    "recall": 2,
    "f1-score": 2
})

report_df

,precision,recall,f1-score,support
Not Success (0),0.92,0.78,0.85,1096
Success (1),0.39,0.68,0.49,223
accuracy,0.76,0.76,0.76,0
weighted avg,0.83,0.76,0.79,1319


In [31]:
np.save("y_test.npy", np.array(y_test))

In [33]:
np.save("y_pred_v1.npy", np.array(y_pred_prob))